# Stage One - The Baseline Harness

**What this notebook is for.** Stage one builds a measuring instrument and then
checks that the instrument is accurate. No training happens here. By the end you
will have two numbers - BM25 and an off-the-shelf bi-encoder - and one answer to
the question *"is my evaluation code correct?"*

**Why that question comes first.** Your evaluation code fails silently. A wrong
prefix, a wrong nDCG variant, the wrong subset of queries: none of these crash.
They all produce output that looks completely fine and scores a few points low.

That matters because of one thing: **a calibration failure and a real improvement
look identical.** If your baseline is silently four points low, and you happen to
fix the bug during stage two, you will report a four-point "gain from fine-tuning"
that is entirely your own bug.

So you check your number against a published one, measured by someone else on the
same model and the same data. That external check is the only thing that can
catch a silent failure.

---

**How to read this notebook.** Run the cells in order. Each section explains what
we are doing, does it in a small runnable piece, then points at where the
production version lives in `src/`. The `src/` files are what you actually run;
this notebook is how you understand them.

## 0. Setup

Two things happen here. We put `src/` on the import path so the notebook can use
the real project code, and we move the working directory to the project root so
that every relative path (`configs/scifact.yaml`, `data/runs/...`) means the same
thing here as it does on the command line.

In [ ]:
import os
import sys
from pathlib import Path

# Walk up until we find the project root (the folder holding configs/).
here = Path.cwd()
while not (here / "configs").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
sys.path.insert(0, str(here / "src"))

print("project root:", Path.cwd())
print("python      :", sys.version.split()[0])

In [ ]:
# The libraries this notebook uses, and what each is for.
import json
import hashlib
import re

import numpy as np          # arrays of numbers; embeddings are arrays
import faiss                # fast similarity search over those arrays
import pytrec_eval          # the standard IR metrics, wrapping the trec_eval C code
from rank_bm25 import BM25Okapi   # the word-counting baseline

print("faiss", faiss.__version__)

## 1. The config

Every script in this project takes `--config configs/scifact.yaml`. Nothing is
hardcoded. Swapping from SciFact to NFCorpus is a flag, not an edit.

There is a second reason for it that only becomes obvious later. When
`results/metrics.csv` has twelve rows and you cannot remember which learning rate
produced row seven, the config file is the answer, and it is in git.

`src/config.py` is also **the only file allowed to know about model prefixes**.
More on that in section 5.

In [ ]:
from config import load_config

cfg = load_config("configs/scifact.yaml")
print(cfg.summary())

### Reading a config value

`cfg` is a frozen dataclass, so you access fields with a dot. Frozen means it
cannot be modified after loading - a config that changes halfway through a run
would make the record of the experiment a lie.

In [ ]:
print("dataset       ", cfg.dataset)
print("base model    ", cfg.base_model)
print("graded relev. ", cfg.graded_relevance)   # False for SciFact: judgments are 0/1
print("trainable     ", cfg.trainable)          # False for FiQA, which is eval-only
print("seed          ", cfg.seed)

# Try this: cfg.dataset = "nfcorpus"  ->  raises FrozenInstanceError

## 2. The data

SciFact ships three things. Understanding their shapes is most of step 1.

| File | What it holds |
|---|---|
| `corpus.jsonl` | ~5,183 scientific abstracts. Each has `_id`, `title`, `text`. |
| `queries.jsonl` | ~1,109 scientific claims. Each has `_id`, `text`. |
| `qrels/test.tsv` | The answer key: which document answers which query. |

`src/ingest.py` has already downloaded these and written a **frozen copy**. We
read that copy rather than re-downloading.

That is not about saving a download. Your evaluation set is a *decision*, and the
frozen copy is where the decision is written down so it cannot drift. Section 4
shows what drift costs - the short version is that losing 10 of your 300 queries
would raise your score by 2.5 nDCG points for free, which is the same size as the
entire gain fine-tuning is supposed to produce.

In [ ]:
from ingest import doc_text, load_frozen_split, load_raw_corpus

corpus = load_raw_corpus("scifact")
queries, qrels, manifest = load_frozen_split("scifact", split="test")

print(f"{len(corpus):>6} documents")
print(f"{len(queries):>6} test queries")
print(f"{sum(len(v) for v in qrels.values()):>6} relevance judgments")

### Look at an actual document

Do this early and often. A surprising number of pipeline bugs are visible the
moment you print the data.

In [ ]:
doc_id = sorted(corpus)[0]
doc = corpus[doc_id]

print("id   :", doc_id)
print("title:", doc["title"])
print("text :", doc["text"][:300], "...")

### `doc_text` - flattening a document to one string

A document has a title and a body, but a retrieval model wants one string. How
you join them is a real design decision:

- **Title first, joined with a space.** This is the BEIR convention, which
  matters because the published numbers you are checking against were measured
  that way.
- **It is not cosmetic.** A SciFact title carries a lot of the topical signal.
  Dropping it costs real points.
- **Defined in exactly one place.** BM25, the bi-encoder, and any future
  reranker all call this same function. If they disagreed about document text,
  their scores would not be comparable and nothing would tell you.

In [ ]:
print(repr(doc_text(doc)[:200]))

### Look at a query and its answer key

Notice the shape of `qrels`: a dictionary of dictionaries.
`qrels[query_id][doc_id] = relevance`. Only relevant pairs are listed. **Any pair
absent from this file is judged irrelevant by omission** - there is no explicit
list of wrong answers.

In [ ]:
qid = sorted(qrels)[0]
print("query id  :", qid)
print("query text:", queries[qid])
print("judged docs:", qrels[qid])

for did, rel in qrels[qid].items():
    print(f"\n  relevant doc {did} (relevance={rel}):")
    print("   ", corpus[did]["title"])

## 3. Two traps that fail silently

These are the reason stage one exists. Neither raises an exception.

### Trap 1: the ids have different types

In the raw downloads, `query-id` in the qrels comes through as an **integer**,
while `_id` in the queries file is a **string**. Join them naively and you match
nothing - no error, no warning, just an empty result and a score of zero.

In [ ]:
# A miniature version of the bug.
queries_str = {"7": "does eating eggs raise cholesterol"}
qrels_int   = {7: {1: 1}}          # ints, as the raw qrels ship them

matched = [q for q in qrels_int if q in queries_str]
print("matches found:", len(matched))       # 0. And nothing complained.

# The fix, applied at the boundary where data enters, in ingest._sid:
qrels_fixed = {str(q): {str(d): r for d, r in docs.items()}
               for q, docs in qrels_int.items()}
print("matches after casting:", len([q for q in qrels_fixed if q in queries_str]))

`src/ingest.py` casts every id with `_sid()` the moment it is loaded, and
`check_joins()` then asserts that every id in the qrels actually exists in the
corpus and the queries. It crashes loudly if not.

There is a second reason to use strings: some datasets have ids with leading
zeros, and `int()` destroys those irreversibly.

### Trap 2: the queries file is bigger than the evaluation set

`queries.jsonl` holds every query across all splits. The test qrels cover only
the test ones. **Your evaluation set is defined by the qrels, not by the queries
file.** Getting this backwards silently changes what you are measuring.

In [ ]:
from datasets import load_dataset

all_queries = load_dataset(cfg.hf_dataset, "queries")["queries"]
print(f"queries.jsonl holds      {len(all_queries)} queries")
print(f"but the test qrels cover {len(qrels)} of them")
print(f"-> {len(all_queries) - len(qrels)} queries belong to other splits")

## 4. Freezing the split, and what hashing means

### The problem

Every number you produce over the next month has to be comparable to every other
number. If you quietly evaluate on 297 queries in week one and 300 in week three,
the two results look comparable, differ by half a point, and are not comparable
at all. You would never notice.

### How much does that actually cost?

Worth measuring rather than taking on faith. The cell below takes your real dense
run and drops the worst-scoring queries, changing nothing else about the model or
the pipeline.

### What a hash is

Writing the files down records the intention. A hash is what makes it enforceable.

A hash function eats any amount of data and produces a short fixed-length
fingerprint. Two properties make it useful here:

1. The same input always gives the same fingerprint.
2. Change **one byte** anywhere and the fingerprint changes completely.

So a 64-character fingerprint is proof that a file is byte-for-byte what it was
when you recorded it.

In [ ]:
from runfile import read_run

base_run = read_run("data/runs/scifact_base_test.trec")
ev = pytrec_eval.RelevanceEvaluator(qrels, {"ndcg_cut.10"})
per_query = {q: v["ndcg_cut_10"] for q, v in ev.evaluate(base_run).items()}

full = sum(per_query.values()) / len(per_query)
print(f"all {len(per_query)} queries              nDCG@10 = {full:.4f}")

worst = sorted(per_query, key=per_query.get)
for n in (3, 5, 10):
    kept = [q for q in per_query if q not in set(worst[:n])]
    score = sum(per_query[q] for q in kept) / len(kept)
    print(f"dropping the {n:>2} worst queries  nDCG@10 = {score:.4f}"
          f"   (+{100 * (score - full):.2f} points, from nothing)")

print(f"\nqueries scoring exactly 0.0: "
      f"{sum(1 for v in per_query.values() if v == 0.0)} of {len(per_query)}")

Ten queries is 3% of the set, and losing them buys 2.5 nDCG points for free.

Now recall what the guide predicts fine-tuning will gain on NFCorpus: somewhere
between two and six points. **A small change in which queries you score is the
same size as the entire finding your project exists to produce.**

And it would not announce itself. It happens through ordinary work - you add a
reasonable filter next month, or the dataset is updated upstream, or a library
version parses something differently, and the count slips from 300 to 297. The
code runs, the output looks fine, the number moves.

The reason this bites *this* project specifically: every result you will report is
a **difference** between two numbers. Base against fine-tuned. Arm A against
Arm B. FiQA before against after. A difference between two numbers measured on
different query sets is not a result. It is noise in the costume of one. And the
two measurements are weeks apart, so nothing about the second run reminds you
what the first was measured on.

### The fix

Write down exactly which queries and judgments you evaluate on, save them to
files, record a fingerprint of each file, commit them, never touch them again.

In [ ]:
def sha256_of_text(s: str) -> str:
    return hashlib.sha256(s.encode()).hexdigest()

a = sha256_of_text("does eating eggs raise cholesterol")
b = sha256_of_text("does eating eggs raise cholesterol.")   # one extra full stop

print("original :", a)
print("one byte :", b)
print("same?    ", a == b)

### Why the corpus is hashed but not committed

The frozen split lives in `data/frozen_splits/` and **is committed to git**. Its
job is to record decisions *you* made: which queries, which judgments, which dev
sample.

You made no decision about the corpus. It is whatever the dataset ships, and
anyone can re-download the identical thing from the same place. So committing 4MB
of it would add no information. Storing its 64-character fingerprint gives you the
same guarantee - *"the corpus I measured against was this exact one"* - for a few
dozen bytes.

That is the whole trade: **commit your choices, hash everything else.**

In [ ]:
print(json.dumps(manifest, indent=2))

### The contract enforces itself

`load_frozen_split()` re-computes each file's hash and compares it to the
manifest. If a file has drifted since it was frozen, it raises rather than
quietly returning different data. That is what turns the contract from
documentation into something the code actually enforces.

Let us prove it by corrupting a copy.

In [ ]:
import shutil, tempfile
from ingest import _sha256

src_file = Path("data/frozen_splits/scifact/qrels_test.tsv")
tmp = Path(tempfile.mkdtemp()) / "qrels_test.tsv"
shutil.copy(src_file, tmp)

print("copy matches original:", _sha256(tmp) == _sha256(src_file))

with tmp.open("a", encoding="utf-8") as fh:
    fh.write("999\t999\t1\n")          # append one fake judgment

print("after adding one line:", _sha256(tmp) == _sha256(src_file))
print("-> load_frozen_split would refuse to use this file")

### The dev/test split, and the single most important choice in `ingest.py`

You need two evaluation sets:

- **dev** - for every decision. Which learning rate? How many epochs? Look here.
- **test** - looked at once, at the end. This is what you report.

If you tune the learning rate by watching the test score and then report the test
score, you tuned *on* the number you are reporting. The number is no longer an
estimate of how the model does on unseen data, because the data was seen.

**The choice: dev is carved out of the TRAIN split, never out of test.**

The obvious alternative is to split the test set 80/20. Do not. Your stage-one
milestone is reproducing the published number for this model on this dataset, and
that number is measured on the **full** official test set of 300 queries. Slice
the test set and there is nothing left to compare against - you throw away the
only external check you have.

SciFact ships no dev split, so `ingest.py` samples 100 train queries with a
recorded seed and **removes** them from train. Removed, not copied: a query in
both dev and train is a leak, because you would be tuning on data the model was
trained on.

In [ ]:
print("dev provenance:", manifest["dev_provenance"])
print("counts:", json.dumps(manifest["counts"], indent=2))

_, dev_qrels, _ = load_frozen_split("scifact", split="dev")
print("\ndev and test overlap:", len(set(dev_qrels) & set(qrels)), "queries")

## 5. BM25 - the word-counting baseline

BM25 scores a document by the query terms it shares, with three refinements over
naive counting:

- **Rare terms count more.** A shared "cholesterol" is far more informative than
  a shared "the".
- **Repetition saturates.** The tenth occurrence of a word adds much less than
  the second.
- **Long documents are penalised.** Otherwise they win by sheer surface area.

No training, no GPU, seconds to run. It is a genuinely strong baseline and you
should be slightly worried if your neural model does not beat it.

In [ ]:
from bm25 import tokenize

print(tokenize("Does eating EGGS raise cholesterol?  (a study, 2019)"))

**Design choice: the tokenizer is deliberately simple** - lowercase, split on
non-alphanumerics, no stemming, no stopword list.

The reasoning: we are using `rank_bm25` (pure Python) rather than Pyserini (needs
Java). `rank_bm25` will *not* reproduce the published BEIR BM25 numbers because
Anserini tokenises and stems differently. Since we cannot match them anyway, a
half-hearted stemmer would make the gap harder to explain rather than smaller.

**This obligates you to say so in the write-up.** Report your BM25 as
self-consistent rather than leaderboard-comparable, and never put it next to a
published figure as though they were measured the same way.

In [ ]:
# Build a small BM25 index over the first 2000 documents, for speed.
subset_ids = sorted(corpus)[:2000]
tokenized = [tokenize(doc_text(corpus[d])) for d in subset_ids]
bm25 = BM25Okapi(tokenized)

demo_query = "cholesterol levels in blood"
scores = bm25.get_scores(tokenize(demo_query))
top = np.argsort(-scores)[:3]

print("query:", demo_query, "\n")
for rank, i in enumerate(top, 1):
    print(f"{rank}. score={scores[i]:.2f}  {corpus[subset_ids[i]]['title'][:80]}")

### BM25's blind spot, demonstrated

This is the failure the whole project is built around. If the relevant document
says *"dietary cholesterol and serum LDL concentrations"* and the query says
*"eggs"*, there is **no shared word**, so BM25 scores it zero. It cannot find
what it cannot match literally.

In [ ]:
# A tiny corpus. Document 0 is the one that genuinely answers "do eggs raise
# cholesterol". The others are filler, so that rare-term weighting behaves
# normally rather than degenerating on a single-document corpus.
mini_corpus = [
    "dietary cholesterol and serum LDL concentrations in adults",
    "soil pH and crop yield in temperate climates",
    "machine translation for low resource languages",
    "the effect of sleep duration on adolescent attention",
]
mini = BM25Okapi([tokenize(d) for d in mini_corpus])

for q in ["cholesterol", "eggs"]:
    score = mini.get_scores(tokenize(q))[0]      # score against document 0
    shared = set(tokenize(q)) & set(tokenize(mini_corpus[0]))
    print(f"query {q!r:<15} -> BM25 score {score:6.3f}   shared words: {shared or '{}'}")

## 6. Evaluation - what nDCG@10 actually computes

You return 10 documents. Counting how many are relevant throws away something
important: three relevant documents at positions 1, 2, 3 is much better than the
same three at 8, 9, 10. The metric has to be **position-aware**. That is the whole
idea. Everything else is bookkeeping.

The name unpacks backwards:

- **G**ain - the relevance grade of the document at that rank. 1 or 0 for
  SciFact; graded for NFCorpus.
- **D**iscounted - divide by `log2(rank + 1)`. Rank 1 divides by 1, rank 10 by
  about 3.5. A logarithm rather than the rank itself because dividing by rank is
  too harsh and overstates how quickly users lose interest. This is an empirical
  choice, not something derived.
- **C**umulative - sum those across the 10 positions. That is DCG@10.
- **n**ormalized - divide by the DCG of the *best possible* ordering for that
  query, so the result sits in [0, 1] and each query is scored against its own
  ceiling.

Let us compute it by hand for three relevant documents returned at ranks 1, 4
and 8.

In [ ]:
import math

def dcg(relevances):
    """Sum of gain / log2(rank + 1), with rank starting at 1."""
    return sum(rel / math.log2(rank + 1) for rank, rel in enumerate(relevances, start=1))

# Our ranking: relevant at positions 1, 4, 8 (1-indexed).
ours  = [1, 0, 0, 1, 0, 0, 0, 1, 0, 0]
# The best possible ordering of the same three relevant documents.
ideal = [1, 1, 1, 0, 0, 0, 0, 0, 0, 0]

print(f"your DCG  = {dcg(ours):.2f}")
print(f"ideal DCG = {dcg(ideal):.2f}")
print(f"nDCG@10   = {dcg(ours) / dcg(ideal):.2f}")

### Why normalising matters

Some queries have one relevant document, others twenty. Without normalising, the
twenty-relevant query would dominate the average purely because more gain was
available to collect.

It also has a consequence worth knowing: **nDCG@10 can reach 1.0 while you missed
relevant documents.** With 30 relevant documents and only 10 slots, the ideal is
computed over the best 10. nDCG@10 asks *"did you order your top 10 well?"*, not
*"did you find everything?"*

### Why we do not use the function above

**Design choice: `pytrec_eval`, never a hand-written nDCG.**

Several DCG variants exist, differing mainly in whether gain is used raw (as
above) or as `2^gain - 1`. On binary data they agree. On graded data like NFCorpus
they do not. `pytrec_eval` wraps the original `trec_eval` C implementation, which
is the variant BEIR reports.

Write your own and you will eventually lose a day to a discrepancy that lives in
your metric rather than in your pipeline. Here is the same worked example through
the real thing.

In [ ]:
# pytrec_eval wants qrels as {query_id: {doc_id: relevance}}
#                and a run as {query_id: {doc_id: score}}
demo_qrels = {"q": {"rel_a": 1, "rel_b": 1, "rel_c": 1}}
ranking = ["rel_a", "x1", "x2", "rel_b", "x3", "x4", "x5", "rel_c", "x6", "x7"]
demo_run = {"q": {d: float(10 - i) for i, d in enumerate(ranking)}}

evaluator = pytrec_eval.RelevanceEvaluator(demo_qrels, {"ndcg_cut.10"})
print(evaluator.evaluate(demo_run)["q"]["ndcg_cut_10"])   # matches our 0.82

### Recall@100 - the number that decides whether stage three is worth building

Of the genuinely relevant documents, how many made the top 100? This separates
the project's two failure modes:

| | What happened | Can a reranker fix it? |
|---|---|---|
| **A** | The right document sits at rank 800. Never made the shortlist. | No |
| **B** | The right document is at rank 7. In the pile, wrong position. | Yes |

- **Recall@100 around 0.94** - the right answers are in the pile and just need
  reordering. A reranker has real room to work.
- **Recall@100 around 0.55** - half the right answers never made the shortlist.
  No amount of reranking recovers them; the problem is upstream.

You get this number for free. Knowing which situation you are in is itself a
good finding.

## 7. The bi-encoder, and the prefix

A **bi-encoder** converts each document into a fixed list of numbers - 384 of them
for `bge-small-en-v1.5` - ahead of time. A point in 384-dimensional space. At
query time it converts the query to a point and finds the nearest document points.
Milliseconds.

It is fast precisely *because* it summarised each document blind, before knowing
what would be asked of it. That is also its ceiling.

**BGE** stands for BAAI General Embedding. It is a family of bi-encoder models,
and `bge-small-en-v1.5` is the specific one this project uses: 33M parameters,
384 dimensions, 512 max tokens. When the guide says "bi-encoder" and when it says
"BGE", it means this same model.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(cfg.base_model)
model.max_seq_length = cfg.max_seq_length

vec = model.encode("does eating eggs raise cholesterol", normalize_embeddings=True)
print("shape:", vec.shape)          # (384,)
print("first 8 numbers:", np.round(vec[:8], 4))
print("length of the vector:", round(float(np.linalg.norm(vec)), 6))   # 1.0

### Why the vector length is exactly 1.0

**Design choice: normalise to unit length, then use inner product.**

We asked for `normalize_embeddings=True`, which scales every vector to length 1.
For unit vectors, the inner product (multiply element-wise, sum) is *exactly*
cosine similarity. FAISS has a fast exact inner-product index, so normalising once
at build time is cheaper and less error-prone than dividing by norms at every
query.

This creates one rule you must not break: **query vectors have to be normalised
the same way.** If either side is not unit length, the numbers are not cosine
similarities and rankings drift with no error message.

In [ ]:
def cosine(a, b):
    return float(np.dot(a, b))          # valid ONLY because both are unit length

query   = model.encode("does eating eggs raise cholesterol", normalize_embeddings=True)
related = model.encode("dietary cholesterol and serum LDL concentrations",
                       normalize_embeddings=True)
unrelated = model.encode("soil pH and crop yield in temperate climates",
                         normalize_embeddings=True)

print(f"query vs related   : {cosine(query, related):.3f}")
print(f"query vs unrelated : {cosine(query, unrelated):.3f}")

That is the whole point of the project in two numbers. BM25 scored the "related"
passage **zero** against the word "eggs" because they share no words. The
bi-encoder sees them as close, because it works in meaning-space rather than
word-space.

### The prefix trap

BGE was trained expecting a specific instruction prepended to **queries** and
nothing prepended to **documents**:

```
"Represent this sentence for searching relevant passages: "
```

Omit it and the model works. It just works *worse*, with no error, no warning,
no crash. A different model family has a different string - E5 uses `"query: "`
and `"passage: "` - so this is a per-model fact you have to look up and get right.

**Design choice: exactly one place in the codebase attaches a prefix.**

`src/config.py` exposes `for_query()` and `for_document()`. No other file reads
`cfg.query_prefix` directly. That is why the guide calls it "the only file that
knows about prefixes" - it makes the trap structurally impossible rather than
merely documented.

In [ ]:
print(repr(cfg.for_query("does eating eggs raise cholesterol")))
print(repr(cfg.for_document("dietary cholesterol and serum LDL")))

A useful habit: check the claim rather than trusting it. We ran the pipeline both
ways and recorded both rows in `metrics.csv`. Section 9 shows what happened, and
it was not what the guide predicted.

## 8. The index, and retrieval

Once every document is a point in 384-dimensional space, retrieval is "find the
nearest points". FAISS does that.

**Stage one builds only `IndexFlatIP`: exact, brute-force, no approximation.**

That is deliberate. Flat search compares the query against every document, so it
is the *correctness reference* every approximate index gets measured against. At
SciFact's 5,183 documents the index is about 8MB and searches in well under a
millisecond - an approximate index there would be solving a problem that does not
exist. The ANN sweep belongs on FiQA, at 58k documents, where it is honest.

Knowing when **not** to reach for a technique reads better than using it
everywhere.

In [ ]:
stem = Path("data/embeddings/scifact_base")
index = faiss.read_index(str(stem.with_suffix(".faiss")))
doc_ids = json.loads(stem.with_suffix(".ids.json").read_text())
meta = json.loads(stem.with_suffix(".meta.json").read_text())

print(json.dumps(meta, indent=2))

### `doc_ids` is not optional bookkeeping

FAISS stores vectors in rows and hands back **row numbers**, not document ids. The
`doc_ids` list is the only thing that maps a search result back to a document.
Row `i` of the index is `doc_ids[i]`, and nothing may reorder one without the
other. That is why `build_index.py` sorts the ids once and saves the list right
next to the index.

In [ ]:
def search(query_text, k=5, show=True):
    """Encode one query, search the index, return the ranked document ids."""
    q_vec = model.encode([cfg.for_query(query_text)],
                         normalize_embeddings=True).astype(np.float32)
    scores, positions = index.search(q_vec, k)
    ranked = [doc_ids[p] for p in positions[0]]      # row number -> document id
    if show:
        for rank, (did, score) in enumerate(zip(ranked, scores[0]), start=1):
            print(f"{rank}. {score:.3f}  {corpus[did]['title'][:70]}")
    return ranked


demo_qid = sorted(qrels)[0]
print("query:", queries[demo_qid][:100], "\n")
search(queries[demo_qid])

Are any of those actually right? The answer key says which documents count. A
useful habit is to report **where the correct document landed**, not just what
came back - the rank is the thing nDCG is built out of.

In [ ]:
def rank_of_relevant(query_id, k=100):
    """Find where each judged-relevant document appears in our ranking."""
    ranked = search(queries[query_id], k=k, show=False)
    for did in qrels[query_id]:
        position = ranked.index(did) + 1 if did in ranked else None
        yield did, position


for did, position in rank_of_relevant(demo_qid):
    where = f"rank {position}" if position else f"not in the top 100"
    print(f"relevant doc {did}: {where}")
    print(f"   {corpus[did]['title'][:80]}")

Run that over all 300 queries and average the position-weighted score and you
have nDCG@10. That is all the metric is: this loop, discounted by rank, averaged.

### The run file - the one interface that matters

Every retrieval system in this project writes the same TREC-format file:

```
query_id  Q0  doc_id  rank  score  run_name
```

BM25 writes it. The bi-encoder writes it. A fine-tuned model writes it. A stage
three reranker reads one and writes another. The consequences are worth spelling
out:

- Comparing four systems is four run files against **one** call to `evaluate.py`,
  not four code paths.
- `evaluate.py` never imports a model, so it cannot accidentally re-encode
  anything and silently disagree with what retrieval actually did.
- You can hand someone `data/runs/` on its own and they can reproduce every number
  in your report **without your model checkpoints**.

The `Q0` column is a vestige of the original TREC format. It carries no
information and is always the literal string `Q0`.

In [ ]:
print(Path("data/runs/scifact_base_test.trec").read_text().split("\n")[0])

Two small details in `runfile.write_run` that are easy to miss:

- **Rank is assigned on write, from score.** It is never taken from the caller.
  That way rank can never disagree with score, and a reranker that only produces
  new scores cannot leave a stale rank column behind.
- **Ties break on document id.** Without a tiebreak, equal scores order
  arbitrarily and two runs from identical inputs can differ. You would chase a
  regression that does not exist.

## 9. Reading the results

Here is what stage one actually produced. `results/metrics.csv` is append-only:
one row per experiment, never overwritten. By the end of the project it is the
single source of truth from which every table and plot in the write-up is
generated. Overwriting a row would destroy the comparison the project exists to
make - you cannot report a base-to-tuned delta if the base row was replaced.

In [ ]:
import pandas as pd

df = pd.read_csv("results/metrics.csv")
df[["model_name", "index_type", "ndcg@10", "recall@100", "query_prefix_used", "notes"]]

### What these numbers say

**The milestone passed.** `bge-small-en-v1.5` scored **0.7127** nDCG@10 on
SciFact. The published MTEB figure is about 0.713. Within a tenth of a point, so
the harness is measuring what it claims to measure. *Verify this against the
current MTEB leaderboard yourself - the figures shift as it is recomputed.*

**BM25 at 0.6523** is close to the published Anserini figure of about 0.665. The
gap is the tokenizer difference discussed in section 5, and it is why you report
this as self-consistent rather than leaderboard-comparable.

**Recall@100 of 0.945** means the right answers are almost always in the top 100.
That is failure mode B, so a stage three reranker has real headroom here.

**The prefix ablation contradicts the guide, and that is a genuine finding.**
The guide predicts that omitting the prefix costs about four nDCG points. It cost
essentially nothing: 0.7132 without versus 0.7127 with.

Before believing that, we checked it was not a bug - no query has the same top-10
ordering across the two runs, so the prefix really was applied and really did
change the rankings. Two things explain it. BAAI's own model card says the v1.5
release was trained to work well *without* the instruction, and SciFact queries
are long declarative claims rather than short keyword queries, which is the case
where the instruction helps most.

**This is what the harness is for.** You measured a claim instead of assuming it,
and got an answer you can defend. Re-run the same ablation on NFCorpus before
concluding anything general - that is one command now.

## 10. Where to go next

Stage one is done when your dense number matches a published one. It does.

**Week 2** is error analysis and the index sweep. Read 20 queries where BM25 beats
the dense model - actually read them, the pattern is usually obvious and it will
shape stage two. Then do the ANN sweep on FiQA, and try hybrid retrieval with
reciprocal rank fusion. That is an afternoon and it is what production systems
actually do.

**Week 3** is stages 2's real content: synthetic query generation, hard negative
mining, contrastive fine-tuning, and the two-arm comparison that is the headline
finding.

### The command that repeats everything on a new dataset

Nothing in the code changes. That is what the config was for.

```
make stage1 CONFIG=configs/nfcorpus.yaml
```

One warning: encoding 5,183 SciFact documents took about nine minutes on your CPU.
FiQA at 58k documents will take roughly two hours locally. That is where you move
to Colab and a GPU.

### Where the real code lives

| File | Does |
|---|---|
| `src/config.py` | loads YAML, owns the prefixes |
| `src/ingest.py` | downloads, freezes the split, hashes it |
| `src/runfile.py` | reads and writes TREC run files |
| `src/bm25.py` | the lexical baseline |
| `src/build_index.py` | encodes the corpus, builds the FAISS index |
| `src/retrieve.py` | encodes queries, searches, writes a run file |
| `src/evaluate.py` | run file + qrels -> a row in metrics.csv |

Run the tests whenever you change any of them:

```
.venv\Scripts\python -m pytest tests/ -q
```